# 💳 Project Task: GoPay Fintech Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
GoPay telah berkembang menjadi tulang punggung ekosistem Super App GoTo. Fitur GoPayLater — layanan kredit berbasis limit — berhasil mendongkrak GTV, namun kini menghadapi masalah serius: tingkat NPL (Non-Performing Loan) yang meningkat, bug validasi limit kredit, dan inkonsistensi data dari puluhan micro-service.

Kamu berperan sebagai Data Analyst di tim **Risk Management GoPay** yang diminta untuk membersihkan data, mengidentifikasi pola gagal bayar, dan memberikan rekomendasi perbaikan credit scoring.

**Dataset (3 tabel):**
- `gopay_users.csv` — 35.000 baris (Dimensi User)
- `gopay_services.csv` — 20 baris (Dimensi Layanan)
- `gopay_transactions.csv` — 300.000 baris (Fakta Transaksi)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> Lebih dari **50% transaksi GoPayLater** memiliki `amount` yang **melebihi `paylater_limit`** user yang bersangkutan.  
> Ini adalah bug sistematis pada validasi limit kredit — bukan sekadar outlier biasa.  
> Identifikasi, kuantifikasi dampak finansialnya, dan rekomendasikan perbaikan.

---
## 0. Import & Load Data

In [572]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [573]:
# Load semua dataset
# Sesuaikan path dengan lokasi file kamu
df_users_raw    = pd.read_csv('dataset/gopay_users.csv')
df_services_raw = pd.read_csv('dataset/gopay_services.csv')
df_trx_raw      = pd.read_csv('dataset/gopay_transactions.csv')

# Buat copy untuk dikerjakan
df_users    = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx      = df_trx_raw.copy()

print(f'users       : {df_users.shape}')
print(f'services    : {df_services.shape}')
print(f'transactions: {df_trx.shape}')

users       : (35000, 5)
services    : (20, 3)
transactions: (300000, 8)


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [574]:
# Shape dan info umum — lakukan untuk ketiga tabel

display(df_users_raw,
df_services_raw,
df_trx_raw)

df_users = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx = df_trx_raw.copy()

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
0,GP-000001,2022-05-26,Basic,NaN,0
1,GP-000002,2023-07-12,Plus,751.00,500000
2,GP-000003,2022-05-09,Plus,728.00,1500000
3,GP-000004,2021-08-28,Basic,394.00,0
4,GP-000005,2021-05-31,Basic,333.00,0
...,...,...,...,...,...
34995,GP-034996,2022-03-12,Plus,544.00,1500000
34996,GP-034997,2023-07-23,Plus,NaN,5000000
34997,GP-034998,2023-08-06,Plus,665.00,1500000
34998,GP-034999,2022-05-29,Basic,391.00,0


,service_id,service_name,category
0,SVC-001,GoRide,Mobility
1,SVC-002,GoCar,Mobility
2,SVC-003,GoBluebird,Mobility
3,SVC-004,GoFood,Food Delivery
4,SVC-005,GoMart,Food Delivery
5,SVC-006,GoSend,Logistics
6,SVC-007,GoBox,Logistics
7,SVC-008,Pulsa/Data,Digital Goods & Bills
8,SVC-009,PLN,Digital Goods & Bills
9,SVC-010,PDAM,Digital Goods & Bills


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid
...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid


In [575]:
# Tipe data seluruh kolom
df_users_raw.info()
df_services_raw.info()
df_trx_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                35000 non-null  str    
 1   join_date              35000 non-null  str    
 2   gopay_tier             35000 non-null  str    
 3   internal_credit_score  29750 non-null  float64
 4   paylater_limit         35000 non-null  int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 1.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   service_id    20 non-null     str  
 1   service_name  20 non-null     str  
 2   category      20 non-null     str  
dtypes: str(3)
memory usage: 612.0 bytes
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  

In [576]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values
display(df_users_raw.isnull().sum())
display(df_services_raw.isnull().sum())
display(df_trx_raw.isnull().sum())

na_values = df_users.isnull().sum()
print('Total nilai kosong pada tiap kolom')
na_values.apply(lambda x : f'{x} - {x/len(df_users):.2%}')

user_id                     0
join_date                   0
gopay_tier                  0
internal_credit_score    5250
paylater_limit              0
dtype: int64

service_id      0
service_name    0
category        0
dtype: int64

trx_id            0
user_id           0
service_id        0
trx_date          0
payment_method    0
amount            0
late_fee          0
payment_status    0
dtype: int64

Total nilai kosong pada tiap kolom


user_id                      0 - 0.00%
join_date                    0 - 0.00%
gopay_tier                   0 - 0.00%
internal_credit_score    5250 - 15.00%
paylater_limit               0 - 0.00%
dtype: str

In [577]:
# Distribusi kolom-kolom kritis
# payment_method, amount, late_fee, payment_status, gopay_tier, internal_credit_score
display
(df_trx['amount'].describe(),
df_trx['payment_method'].describe(),
df_trx['late_fee'].describe(),
df_trx['payment_status'].describe()
)

(count    300000.00
 mean     447504.65
 std      490747.56
 min       15004.00
 25%      213294.75
 50%      413388.50
 75%      612431.75
 max     9965039.00
 Name: amount, dtype: float64,
 count     300000
 unique         8
 top        GoPay
 freq       90240
 Name: payment_method, dtype: object,
 count     300000.00
 mean       26734.99
 std      1591437.75
 min       -15000.00
 25%            0.00
 50%            0.00
 75%            0.00
 max     99999999.00
 Name: late_fee, dtype: float64,
 count     300000
 unique         3
 top         Paid
 freq      273921
 Name: payment_status, dtype: object)

In [578]:
display(
df_users['gopay_tier'].describe(),
df_users['internal_credit_score'].describe()
)

count     35000
unique        2
top        Plus
freq      21062
Name: gopay_tier, dtype: object

count   29750.00
mean      534.25
std       153.50
min       300.00
25%       415.00
50%       490.00
75%       662.00
max       849.00
Name: internal_credit_score, dtype: float64

In [579]:
# Cek konsistensi relasi antar tabel
# Apakah semua service_id di transactions ada di services?
# Apakah semua user_id di transactions ada di users?
display(
    df_trx[
        ~df_trx['user_id'].isin(df_users['user_id'])
    ],
    df_trx[
        ~df_trx['service_id'].isin(df_services['service_id'])
    ]
)

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


**✍️ Ringkasan Temuan Eksplorasi:**

*(Kolom apa yang bermasalah di setiap tabel, seberapa parah, dan prioritas penanganan kamu)*

> 

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | Sebagian kecil `internal_credit_score` kosong karena gangguan sistem scraping acak. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `internal_credit_score` kosong mungkin berkorelasi dengan `gopay_tier` atau `join_date`. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `internal_credit_score` kosong justru karena user tidak pernah bertransaksi — nilai kosong itu sendiri adalah sinyal risiko. |

> 💡 **Cara Menggunakan Kerangka Ini:**
> Untuk setiap kolom bermasalah, tanyakan:
> 1. Apakah pola missing-nya acak, atau ada pola tertentu?
> 2. Apakah nilai kosong berkaitan dengan kolom lain?
> 3. Apakah nilai kosong itu sendiri mengandung informasi bisnis?
>
> Justifikasi reasoning kamu lebih penting dari labelnya.

In [580]:
display(

df_users[
    (df_users['internal_credit_score'].isnull())
    & (df_users['paylater_limit'] > 0)
    & (df_users['gopay_tier'] == 'Basic')
] # Secara Akun Basic sudah benar internal_credit_score null karena paylater_limit = 0  (benar)


,df_users[
    (df_users['paylater_limit'] > 0)
    & (df_users['gopay_tier'] == 'Basic')
] # Secara Akun Basic tidak ada yang PayLater Limit > 0  (benar)

,df_users[
    (df_users['internal_credit_score'].notnull())
    & (df_users['gopay_tier'] == 'Basic')
] # Secara Akun Basic seharusnya internal_credit_score = 0 (Masalah)

,df_users[
    (df_users['internal_credit_score'].isnull())
    & (df_users['paylater_limit'] > 0)
    & (df_users['gopay_tier'] == 'Plus')
] # Akun Plus Seharusnya memiliki credi score semua (masalah)

,df_users[
    (df_users['paylater_limit'] == 0)
    & (df_users['gopay_tier'] == 'Plus')
] #  Akun Plus semua memiliki Payment_Limit yang tidak 0 (benar)
)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
3,GP-000004,2021-08-28,Basic,394.00,0
4,GP-000005,2021-05-31,Basic,333.00,0
11,GP-000012,2022-02-05,Basic,333.00,0
12,GP-000013,2023-01-04,Basic,405.00,0
13,GP-000014,2023-02-20,Basic,339.00,0
...,...,...,...,...,...
34989,GP-034990,2021-06-08,Basic,497.00,0
34992,GP-034993,2021-11-15,Basic,493.00,0
34994,GP-034995,2022-05-05,Basic,367.00,0
34998,GP-034999,2022-05-29,Basic,391.00,0


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
7,GP-000008,2023-03-26,Plus,NaN,500000
9,GP-000010,2021-04-20,Plus,NaN,5000000
19,GP-000020,2022-06-23,Plus,NaN,3000000
25,GP-000026,2022-01-15,Plus,NaN,500000
35,GP-000036,2022-07-20,Plus,NaN,1500000
...,...,...,...,...,...
34973,GP-034974,2022-06-14,Plus,NaN,3000000
34977,GP-034978,2022-04-26,Plus,NaN,500000
34980,GP-034981,2023-02-28,Plus,NaN,1500000
34988,GP-034989,2021-03-16,Plus,NaN,1500000


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


---
### 2.3 Penanganan Kolom `payment_method`

Kolom ini memiliki 8 varian penulisan untuk 3 metode pembayaran yang berbeda, akibat inkonsistensi penamaan antar micro-service.

| Varian Asli | Metode Sebenarnya |
|---|---|
| `GoPay`, `gopay`, `GO-PAY` | GoPay (saldo digital) |
| `GoPayLater`, `PayLater`, `gopay_later` | GoPayLater (kredit) |
| `Cash`, `CASH` | Cash |

> 🧠 **Critical Thinking Prompt:**  
> Setelah standarisasi, periksa ulang: apakah ada user **Basic tier** yang menggunakan GoPayLater?  
> Secara aturan bisnis, PayLater hanya boleh digunakan oleh user Plus.  
> Jika ada, apakah itu error data atau bug sistem validasi?

In [581]:
# Lihat semua nilai unik di payment_method beserta frekuensinya
df_trx['payment_method'].unique()

<StringArray>
[      'GoPay',       'gopay',    'PayLater',  'GoPayLater',      'GO-PAY',
        'Cash', 'gopay_later',        'CASH']
Length: 8, dtype: str

**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Format standar yang kamu pilih dan alasannya:
- Temuan setelah standarisasi (apakah ada Basic tier yang pakai GoPayLater?):

> 

In [582]:
# TODO: Standarisasi payment_method
# Simpan hasil ke kolom baru: payment_method_clean
def normalize_payment_method(x):
    if x['payment_method'] in ['gopay', 'GO-PAY', 'GoPay']:
        return 'GoPay'
    elif x['payment_method'] in ['GoPayLater', 'PayLater', 'gopay_later']:
        return 'GoPayLater'
    elif x['payment_method'] in ['Cash', 'CASH']:
        return 'Cash'

df_trx_clean = df_trx.copy()

df_trx_clean['payment_method'] = df_trx_clean.apply(normalize_payment_method, axis=1)
df_trx_clean

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,GoPay,34229,0.00,Paid
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,GoPay,234183,0.00,Paid
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,GoPayLater,380431,0.00,Paid
...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,Cash,327414,0.00,Paid
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid


In [674]:
# Verifikasi: cek Basic tier yang menggunakan GoPayLater setelah standarisasi
user_trx = df_users.merge(df_trx_clean,'inner',on='user_id')
user_trx

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid
1,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid
2,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid
3,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid
4,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default
...,...,...,...,...,...,...,...,...,...,...,...,...
153703,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583,0.00,Paid
153704,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,GoPay,708976,0.00,Paid
153705,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240,0.00,Paid
153706,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077,13120.00,Default


In [584]:
user_trx[
    (user_trx['payment_method'] =='GoPayLater')
    &(user_trx['gopay_tier'] == 'Basic')
]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928,0.00,Paid
33,GP-000005,2021-05-31,Basic,333.00,0,GTRX-0026167,SVC-003,2023-03-07 07:00:00,GoPayLater,195639,0.00,Paid
34,GP-000005,2021-05-31,Basic,333.00,0,GTRX-0089251,SVC-014,2023-02-08 13:00:00,GoPayLater,659008,0.00,Paid
36,GP-000005,2021-05-31,Basic,333.00,0,GTRX-0230522,SVC-003,2023-08-27 10:00:00,GoPayLater,367491,0.00,Paid
87,GP-000012,2022-02-05,Basic,333.00,0,GTRX-0090932,SVC-016,2023-01-03 20:00:00,GoPayLater,154838,0.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...
299988,GP-034999,2022-05-29,Basic,391.00,0,GTRX-0283795,SVC-005,2023-06-03 07:00:00,GoPayLater,648804,0.00,Paid
299990,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0039481,SVC-007,2023-09-13 08:00:00,GoPayLater,643625,22789.00,Default
299991,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0050933,SVC-017,2023-01-17 16:00:00,GoPayLater,222357,10905.00,Default
299997,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400,0.00,Paid


---
### 2.4 Penanganan `internal_credit_score`

Kolom `internal_credit_score` di tabel users memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

> 🧠 **Critical Thinking Prompt:**  
> Apakah nilai kosong ini karena sistem gagal mencatat, atau karena user memang belum punya histori kredit?  
> User tanpa credit score = *unscored* — di industri fintech, ini dianggap risiko tersendiri.  
> Keputusan kamu di sini akan langsung mempengaruhi hasil analisis profil risiko di Section 4.

In [585]:
# Investigasi pola missing values pada internal_credit_score
# Apakah berkorelasi dengan gopay_tier, paylater_limit, atau join_date?
display(

user_trx[
    user_trx['internal_credit_score'].isnull()
],
user_trx[
    user_trx['internal_credit_score'].notnull()
]
)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068,0.00,Paid
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GoPay,195808,0.00,Paid
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,GoPay,606859,0.00,Paid
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899,0.00,Paid
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928,0.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...
299961,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0093771,SVC-003,2023-04-27 12:00:00,GoPay,230848,0.00,Paid
299962,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0222554,SVC-009,2023-02-24 21:00:00,GoPay,294985,0.00,Paid
299963,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0265566,SVC-002,2023-01-06 22:00:00,GoPayLater,293931,0.00,Paid
299964,GP-034997,2023-07-23,Plus,NaN,5000000,GTRX-0275258,SVC-001,2023-11-21 21:00:00,GoPay,304070,0.00,Paid


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650,0.00,Paid
299996,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,GoPay,138743,0.00,Paid
299997,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400,0.00,Paid
299998,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,GoPay,568217,0.00,Paid


### Berdasarkan data perbandingan diatas mendapat kesimpulan missing value pada internal_credit_score tidak berhubungan dengan (gopay_tier, paylater_limit, atau join_date)

In [586]:
# Cek: apakah user yang credit_score-nya missing lebih banyak yang default?
# Hint: merge dengan df_trx, lalu bandingkan default rate

user_trx.loc[
    user_trx['internal_credit_score'].isnull(),
    'payment_status'
].value_counts(normalize=True) * 100


payment_status
Paid      91.33
Default    5.07
Pending    3.60
Name: proportion, dtype: float64

**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi pola missing (berkorelasi dengan tier? join_date?):
- Apakah user tanpa credit score memiliki default rate yang berbeda?
- Keputusan penanganan (drop / impute / pertahankan NaN) dan alasan:

> 

In [670]:
# TODO: Implementasi penanganan missing values internal_credit_score
display(
df_users[
    (df_users['gopay_tier'] == 'Basic')
    &(df_users['internal_credit_score'].isnull())
]
,df_users[
    (df_users['gopay_tier'] == 'Plus')
    &(df_users['internal_credit_score'].isnull())
]
)


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,date_anomaly


In [669]:
df_users = df_users[
    (df_users['gopay_tier'] == 'Plus')
    &(df_users['internal_credit_score'].notnull())
]

user_trx = user_trx[
    (user_trx['gopay_tier'] == 'Plus')
    &(user_trx['internal_credit_score'].notnull())
]

df_users[
    (df_users['gopay_tier'] == 'Plus')
    &(df_users['internal_credit_score'].isnull())
]



,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


Penanganan missing value internal_credit_score adalah dipertahankan 
Alasan:
- Tidak terdapat pola tertentu yang menyebabkan kekosongan credit_score
- Karena 2122 rows merupakan Tier Basic, yang masuk akal bila credit_score nya kosong.
- Sedangkan 3128 rows merupakan Tier Plus, dilakukan penghapusan

---
### 2.5 Penanganan `late_fee`

Kolom `late_fee` memiliki dua jenis anomali yang berbeda sifatnya — tangani secara terpisah.

> 🧠 **Critical Thinking Prompt:**  
> Di industri fintech, denda keterlambatan diatur oleh regulasi OJK.  
> Nilai `late_fee` yang sangat besar bisa berarti bug sistem, bukan kebijakan yang valid.  
> Keputusan kamu harus mempertimbangkan aspek **compliance**, bukan hanya statistik.

In [589]:
# Investigasi distribusi late_fee secara menyeluruh
# Berapa nilai negatif? Berapa nilai ekstrem?
df_trx_clean[
    df_trx_clean['late_fee'] <0
].sort_values('late_fee',ascending=True)


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
1586,GTRX-0001587,GP-010619,SVC-018,2023-12-02 02:00:00,GoPayLater,4196479,-15000.00,Default
7772,GTRX-0007773,GP-023971,SVC-016,2023-10-26 09:00:00,GoPayLater,191304,-15000.00,Default
10931,GTRX-0010932,GP-005728,SVC-012,2023-09-02 19:00:00,GoPayLater,161718,-15000.00,Default
15832,GTRX-0015833,GP-004758,SVC-017,2023-04-13 02:00:00,GoPayLater,622993,-15000.00,Default
15853,GTRX-0015854,GP-015440,SVC-006,2023-02-18 04:00:00,GoPayLater,678788,-15000.00,Default
...,...,...,...,...,...,...,...,...
260771,GTRX-0260772,GP-012434,SVC-020,2023-05-08 05:00:00,GoPayLater,736970,-15000.00,Default
273060,GTRX-0273061,GP-027094,SVC-015,2023-03-29 15:00:00,GoPayLater,372457,-15000.00,Default
276295,GTRX-0276296,GP-009042,SVC-003,2023-09-13 18:00:00,GoPayLater,358111,-15000.00,Default
277153,GTRX-0277154,GP-024238,SVC-001,2023-10-15 16:00:00,GoPayLater,57824,-15000.00,Default


In [666]:
# Anomali 1: Nilai negatif
# Apakah terjadi pada payment_status tertentu?
# display(
# user_trx.loc[
#     user_trx['late_fee'] <0
# ,'payment_status'].unique()

# ,user_trx[
#     (user_trx['payment_status'] == 'Default')
#     & (user_trx['late_fee'] > 0)
# ]


# ,user_trx[
#     (user_trx['payment_status'] == 'Default')
#     & (user_trx['late_fee'] < 0)
# ]
# )


In [591]:
# Anomali 2: Nilai ekstrem tinggi (> Rp 10 juta)
# Apakah ada pola pada service atau user tertentu?
df_trx_clean[
    df_trx_clean['late_fee'] > 10_000_000
].sort_values('late_fee',ascending=False)


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
7841,GTRX-0007842,GP-022226,SVC-008,2023-08-24 15:00:00,GoPayLater,338615,99999999.00,Default
8739,GTRX-0008740,GP-003899,SVC-006,2023-11-23 03:00:00,GoPayLater,776832,99999999.00,Default
12362,GTRX-0012363,GP-025589,SVC-020,2023-03-28 04:00:00,GoPayLater,605600,99999999.00,Default
15722,GTRX-0015723,GP-024468,SVC-019,2023-03-24 02:00:00,GoPayLater,414891,99999999.00,Default
16360,GTRX-0016361,GP-022374,SVC-002,2023-12-08 23:00:00,GoPayLater,311022,99999999.00,Default
...,...,...,...,...,...,...,...,...
285740,GTRX-0285741,GP-006536,SVC-009,2023-03-19 03:00:00,GoPayLater,195821,99999999.00,Default
293899,GTRX-0293900,GP-030129,SVC-014,2023-10-02 17:00:00,GoPayLater,670071,99999999.00,Default
294317,GTRX-0294318,GP-012269,SVC-020,2023-05-26 12:00:00,GoPayLater,149030,99999999.00,Default
294593,GTRX-0294594,GP-004582,SVC-019,2023-08-24 05:00:00,GoPayLater,249567,99999999.00,Default


**✍️ Analisis & Justifikasi — Anomali 1 (late_fee negatif):**
- Jumlah baris terdampak:
- Hipotesis penyebab (logical error? refund denda yang salah catat?):
- Keputusan penanganan dan alasan:

> 

**✍️ Analisis & Justifikasi — Anomali 2 (late_fee ekstrem):**
- Jumlah baris terdampak dan range nilainya:
- Threshold yang kamu pilih untuk mendefinisikan 'ekstrem' dan alasannya:
- Hipotesis penyebab (bug sistem? kebijakan tidak terkontrol?):
- Keputusan penanganan (cap / drop / flag) dan alasan:

> 

In [592]:
# TODO: Implementasi penanganan Anomali 1 (late_fee negatif)
df_trx_clean = df_trx_clean.drop(
    df_trx_clean.loc[df_trx_clean['late_fee'] < 0].index
)

df_trx_clean[
    (df_trx_clean['late_fee'] < 0)
]

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status


In [664]:
# TODO: Implementasi penanganan Anomali 2 (late_fee ekstrem)
df_trx_clean = df_trx_clean.drop(
    df_trx_clean.loc[df_trx_clean['late_fee'] > 10_000_000].index
)

df_trx_clean[
    df_trx_clean['late_fee'] > 10_000_000
].sort_values('late_fee',ascending=False)


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,date_anomaly


---
### 2.6 Penanganan Anomali Tanggal: `trx_date` sebelum `join_date`

Terdapat **~30.177 transaksi (~10%)** dengan `trx_date` lebih awal dari `join_date` user — secara logika bisnis tidak mungkin terjadi.

> 🧠 **Critical Thinking Prompt:**  
> Di konteks fintech, transaksi sebelum akun dibuat bisa mengindikasikan **fraud** atau **data migration issue**.  
> Drop vs. flag memiliki implikasi berbeda: drop menghilangkan sinyal fraud, flag mempertahankannya untuk analisis.  
> Apakah anomali ini lebih banyak terjadi pada user yang akhirnya **Default**?

In [648]:
# Konversi kolom tanggal ke datetime

user_trx['join_date'] = pd.to_datetime(user_trx['join_date'])

user_trx

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,date_anomaly
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid,True
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid,False
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid,True
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid,True
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
299974,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583,0.00,Paid,True
299975,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,GoPay,708976,0.00,Paid,True
299976,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240,0.00,Paid,True
299977,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077,13120.00,Default,False


In [663]:
# Identifikasi transaksi dengan trx_date < join_date
# Investigasi: seberapa besar selisih tanggalnya? Distribusi selisih negatif?
user_trx['date_anomaly'] = (
    user_trx['trx_date'] < user_trx['join_date']
)
user_trx

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,date_anomaly
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid,True
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid,False
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid,True
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid,True
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
299974,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583,0.00,Paid,True
299975,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,GoPay,708976,0.00,Paid,True
299976,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240,0.00,Paid,True
299977,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077,13120.00,Default,False


In [654]:
# Apakah anomali ini berkorelasi dengan payment_status = Default?
# Apakah tersebar merata atau terkonsentrasi pada user/tanggal tertentu?
display(

user_trx.loc[
    user_trx['date_anomaly'] == True
,'join_date'].unique()
,user_trx.loc[
    user_trx['date_anomaly'] == True
,'user_id'].unique()
)


<DatetimeArray>
['2023-07-12 00:00:00', '2023-07-03 00:00:00', '2023-09-04 00:00:00',
 '2023-09-26 00:00:00', '2023-07-22 00:00:00', '2023-03-02 00:00:00',
 '2023-05-01 00:00:00', '2023-03-31 00:00:00', '2023-08-15 00:00:00',
 '2023-03-11 00:00:00',
 ...
 '2023-01-16 00:00:00', '2023-01-19 00:00:00', '2023-03-08 00:00:00',
 '2023-07-24 00:00:00', '2023-07-27 00:00:00', '2023-01-21 00:00:00',
 '2023-01-15 00:00:00', '2023-01-22 00:00:00', '2023-02-02 00:00:00',
 '2023-01-14 00:00:00']
Length: 264, dtype: datetime64[us]

<StringArray>
['GP-000002', 'GP-000011', 'GP-000018', 'GP-000019', 'GP-000041', 'GP-000044',
 'GP-000050', 'GP-000070', 'GP-000075', 'GP-000080',
 ...
 'GP-034956', 'GP-034957', 'GP-034966', 'GP-034970', 'GP-034977', 'GP-034979',
 'GP-034982', 'GP-034984', 'GP-034986', 'GP-034998']
Length: 4088, dtype: str

Data anomali date tersebar dengan rata dan tidak terdapat pola tertentu
terhadap status, user_id, maupun tanggal

**✍️ Analisis & Justifikasi:**
- Jumlah baris terdampak dan distribusi selisih tanggal:
- **Jenis anomali (acak / berpola):** dan alasan klasifikasi kamu:
- Hipotesis penyebab (migration error? clock skew? fraud?):
- Apakah anomali ini berkorelasi dengan Default? Implikasi untuk analisis risiko:
- Keputusan penanganan (drop / flag / pertahankan) dan alasan:

> 

Kemungkinan terjadi akibat migration error, dan akan dipertahankan dengan menambahkan flag date anomaly

---
### 2.7 Penanganan Business Logic Error: Transaksi PayLater Melebihi Limit

**Ini adalah anomali paling kritis di dataset ini.** Lebih dari 50% transaksi GoPayLater memiliki `amount` yang melebihi `paylater_limit` user, termasuk user Basic tier yang seharusnya tidak punya PayLater sama sekali.

| Tipe Pelanggaran | Deskripsi |
|---|---|
| **Basic tier pakai PayLater** | User dengan `paylater_limit = 0` bertransaksi dengan GoPayLater |
| **Plus tier melebihi limit** | User PayLater sah, tapi `amount > paylater_limit` |
| **Transaksi valid** | User Plus dengan `amount ≤ paylater_limit` |

> 🧠 **Critical Thinking Prompt:**  
> Jangan drop transaksi over-limit — ini adalah **data paling berharga** untuk memahami bug dan pola default.  
> Pertahankan dengan flag, lalu analisis secara terpisah.  
> **Dropping = menghilangkan bukti.**

In [660]:
# Merge transaksi GoPayLater dengan data paylater_limit user
# Identifikasi tipe pelanggaran untuk setiap transaksi


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,date_anomaly
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,GoPay,200782,0.00,Paid,True
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279,0.00,Paid,False
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460,0.00,Paid,True
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,GoPay,88206,0.00,Paid,True
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436,37815.00,Default,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
299974,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583,0.00,Paid,True
299975,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,GoPay,708976,0.00,Paid,True
299976,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240,0.00,Paid,True
299977,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077,13120.00,Default,False


In [599]:
# Kuantifikasi: berapa jumlah dan total nilai (Rupiah) dari setiap tipe pelanggaran?


In [600]:
# Kritis: apakah transaksi over-limit berkorelasi dengan payment_status = Default?
# Bandingkan default rate antara: transaksi valid vs over-limit


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase setiap tipe pelanggaran:
- Total nilai Rupiah yang terlibat dalam pelanggaran:
- Apakah over-limit berkorelasi dengan Default? Temuan kamu:
- Keputusan penanganan (flag, bukan drop) dan kolom flag yang kamu buat:
- Hipotesis mengapa bug ini bisa terjadi di sistem:

> 

In [601]:
# TODO: Buat kolom flag untuk tipe pelanggaran PayLater
# Contoh: 'valid', 'over_limit', 'unauthorized'


---
### 2.8 Penanganan Duplikat & Integritas Data

In [602]:
# 1. Cek exact duplicates di setiap tabel


In [603]:
# 2. Cek duplikat trx_id


In [604]:
# 3. Cek service_id di transactions yang tidak ada di services


In [605]:
# 4. Cek inkonsistensi logika: payment_status = Pending tapi late_fee > 0


**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak:
- Hipotesis untuk setiap masalah:
- Keputusan penanganan per masalah:

> 

In [606]:
# TODO: Implementasi keputusan penanganan masalah integritas


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `payment_method_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_trx)*

#### ⚙️ `credit_score_tier`

> 💡 Default threshold: Poor (300–499), Fair (500–649), Good (650–749), Excellent (750–850).  
> Sesuaikan jika analisis distribusi kamu menunjukkan pembagian yang lebih bermakna secara bisnis.

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [607]:
# TODO: Buat credit_score_tier di df_users
# Pertimbangkan: bagaimana menangani user yang credit_score-nya NaN?


#### ⚙️ `is_paylater_violation`

> 💡 Perlu merge df_trx dengan df_users untuk mendapatkan paylater_limit per transaksi.

In [608]:
# TODO: Buat is_paylater_violation (boolean)
# True jika payment_method_clean == 'gopaylater' AND amount > paylater_limit


#### ⚙️ `paylater_usage_ratio`

> 💡 Hanya relevan untuk transaksi GoPayLater. Untuk transaksi non-PayLater, isi dengan NaN.

In [609]:
# TODO: Buat paylater_usage_ratio (amount / paylater_limit)
# Handle division by zero untuk user dengan paylater_limit = 0


#### ⚙️ `user_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

In [610]:
# TODO: Buat user_tenure_days di df_users


#### ⚙️ `has_late_fee`

In [611]:
# TODO: Buat has_late_fee (boolean: True jika late_fee > 0)
# Pastikan menggunakan late_fee yang sudah di-clean dari Section 2.5


#### ⚙️ `is_default`

In [612]:
# TODO: Buat is_default (boolean: True jika payment_status == 'Default')


#### ⚙️ `service_category`

> 💡 Join df_trx dengan df_services untuk mendapatkan kategori layanan per transaksi.

In [613]:
# TODO: Buat service_category dengan merge ke df_services


---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `default_rate_per_user`, `avg_amount_per_service_category`, `is_high_risk_transaction`, `credit_utilization_band`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 1: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [614]:
# TODO: Implementasi Fitur Pilihan 1


#### ⚙️ Fitur Pilihan 2: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [615]:
# TODO: Implementasi Fitur Pilihan 2


---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Transaksi & Metode Pembayaran

**Soal 1:** Berapa total GTV (Gross Transaction Value) keseluruhan? Breakdown GTV per `payment_method_clean`. Metode mana yang paling dominan dan apa implikasi bisnisnya?

In [616]:
# Soal 1


**✍️ Insight:**

> 

**Soal 2:** Berapa distribusi `payment_status` secara keseluruhan? Kemudian breakdown **default rate** per `payment_method_clean`. Apakah GoPayLater memiliki default rate yang lebih tinggi?

In [617]:
# Soal 2


**✍️ Insight:**

> 

**Soal 3:** Berapa rata-rata, median, dan standar deviasi `amount` per `payment_method_clean`? Apa yang bisa disimpulkan dari perbedaan mean vs median?

In [618]:
# Soal 3


**✍️ Insight:**

> 

**Soal 4:** Analisis `late_fee`: Berapa persentase transaksi yang dikenakan denda? Berapa total `late_fee` yang terkumpul? Breakdown per `payment_method_clean`.

In [619]:
# Soal 4


**✍️ Insight:**

> 

---
### 4.2 Analisis Risiko Kredit & Profil User

**Soal 5:** Berapa distribusi `credit_score_tier`? Kemudian bandingkan **default rate** (untuk transaksi GoPayLater) antar `credit_score_tier`. Apakah user dengan credit score rendah memiliki default rate yang lebih tinggi?

In [620]:
# Soal 5
# Hint: merge df_trx (filter GoPayLater) dengan df_users, lalu groupby credit_score_tier


**✍️ Insight:**

> 

**Soal 6:** Berapa persentase `is_paylater_violation = True`? Breakdown antara: Basic tier pakai PayLater vs Plus tier melebihi limit. Berapa total nilai Rupiah yang terlibat?

In [621]:
# Soal 6


**✍️ Insight:**

> 

**Soal 7:** Apakah ada korelasi antara `paylater_usage_ratio` dan `is_default`? Bandingkan rata-rata `paylater_usage_ratio` antara transaksi yang Default vs yang tidak.

In [622]:
# Soal 7


**✍️ Insight:**

> 

**Soal 8:** Berapa distribusi `gopay_tier` di antara user yang pernah Default? Apakah user Basic yang 'membobol' sistem PayLater memiliki default rate lebih tinggi dari user Plus yang sah?

In [623]:
# Soal 8


**✍️ Insight:**

> 

---
### 4.3 Analisis Layanan & Kategori

**Soal 9:** Berapa total GTV dan jumlah transaksi per `service_category`? Kategori mana yang paling tinggi volumenya?

In [624]:
# Soal 9


**✍️ Insight:**

> 

**Soal 10:** Berapa **default rate** per `service_category` untuk transaksi GoPayLater? Layanan mana yang paling berisiko untuk dibayar dengan PayLater?

In [625]:
# Soal 10


**✍️ Insight:**

> 

**Soal 11:** Top 5 `service_name` berdasarkan total `late_fee` yang dikumpulkan. Apakah ini mengindikasikan layanan tertentu lebih sering mengalami keterlambatan pembayaran?

In [626]:
# Soal 11


**✍️ Insight:**

> 

---
### 4.4 Analisis Sistem & Deteksi Anomali *(Implicit — Business Sense Required)*

> Kamu diminta tim **Risk & Compliance** untuk menyusun laporan investigasi sistem PayLater.  
> Temuan ini akan digunakan untuk: (a) menentukan apakah PayLater perlu di-suspend sementara,  
> (b) mengidentifikasi user yang perlu limit adjustment, dan  
> (c) mengestimasi **total kerugian potensial** dari bug yang ada.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Isi judul]

**✍️ Mengapa kamu memilih angle ini untuk investigasi sistem?**

> 

In [627]:
# Angle 1


**✍️ Insight & Rekomendasi untuk Tim Risk & Compliance:**

> 

#### 🔍 Investigasi — Angle 2: [Isi judul]

**✍️ Mengapa kamu memilih angle ini?**

> 

In [628]:
# Angle 2


**✍️ Insight & Rekomendasi:**

> 

---
### 4.5 Credit Risk Profiling *(Implicit — Open Ended)*

> Kamu diminta **Chief Risk Officer GoPay** untuk menyusun rekomendasi perbaikan algoritma credit scoring.  
> Tujuan: menentukan kriteria yang lebih ketat untuk pemberian limit PayLater,  
> sehingga NPL bisa ditekan tanpa terlalu banyak membatasi user yang sebenarnya *creditworthy*.

> 🧠 **Critical Thinking Prompt:**  
> Apakah user dengan credit score rendah **selalu** berisiko?  
> Bagaimana dengan user baru yang belum punya credit score sama sekali?  
> Temukan **sweet spot** antara risk mitigation dan business growth.

**Ekspektasi minimal:**
- Minimal 3 variabel/fitur berbeda yang kamu identifikasi sebagai prediktor default yang signifikan
- Profil 'high-risk user' berdasarkan kombinasi variabel tersebut
- Minimal 1 rekomendasi konkret untuk kebijakan limit PayLater yang berbasis data

**✍️ Definisi 'high-risk user' menurut kamu (dalam konteks kredit GoPay):**

> 

#### 📊 Prediktor Default 1: [Nama Variabel]

In [629]:
# Prediktor 1


#### 📊 Prediktor Default 2: [Nama Variabel]

In [630]:
# Prediktor 2


#### 📊 Prediktor Default 3: [Nama Variabel]

In [631]:
# Prediktor 3


#### 🎯 Profil High-Risk User vs Average User

In [632]:
# Bandingkan karakteristik high-risk user vs keseluruhan user PayLater


**✍️ Rekomendasi Kebijakan Limit PayLater untuk Chief Risk Officer:**

> 

---
## 5. Export Clean Dataset

In [633]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_trx sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_trx.merge(df_users[...], on='user_id', how='left')
#                  .merge(df_services[...], on='service_id', how='left')

# Export
# df_final.to_csv('gopay_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: gopay_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_trx_raw.columns]}')


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait risiko kredit):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Risk Management GoPay:**

> 